# 06 - Inherited Houses Data Preparation

## Introduction

In this notebook, we will process the raw data of the 4 inherited houses provided by the client. The goal is to apply the exact same Data Cleaning and Feature Engineering steps used during the model training phase. This ensures the data is perfectly formatted before being fed into the Streamlit dashboard for price prediction.

## Objectives:

1. **Load the Data**: Import the raw `inherited_houses.csv` file.
2. **Missing Data Handling**: Fill any null values using the same logic applied to the main dataset.
3. **Feature Engineering**: Convert categorical variables (Ordinal Encoding) and calculate age-based features.
4. **Feature Filtering**: Ensure the final dataset columns match exactly the features the model was trained on.
5. **Save Data**: Export the engineered dataset for the dashboard to consume.

---
## 1. Load the Data
We begin by loading the raw records of the inherited houses.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

# Set working directory to project root
full_project_path = "/workspaces/heritage-housing-issues"
os.chdir(full_project_path)

# Load raw inherited houses data
file_path = "inputs/datasets/raw/house-price/inherited_houses.csv"
df_inherited = pd.read_csv(file_path)

print(f"Inherited houses dataset loaded successfully! Shape: {df_inherited.shape}")
df_inherited.head()

Inherited houses dataset loaded successfully! Shape: (4, 23)


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,11622,80.0,0.0,0,6,5,882.0,140,1961,1961
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,14267,81.0,108.0,36,6,6,1329.0,393,1958,1958
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,13830,74.0,0.0,34,5,5,928.0,212,1997,1998
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,9978,78.0,20.0,36,6,6,926.0,360,1998,1998


---
## 2. Missing Data Handling
In this section, we address any missing values within the inherited houses dataset. We will apply the exact same imputation strategies used in Notebook 03 (e.g., filling numerical gaps with the median and categorical gaps with 'None') to maintain data integrity and prevent pipeline errors.

In [2]:
# 1. Drop columns that were excluded during training (>80% missing)
cols_to_drop = ['EnclosedPorch', 'WoodDeckSF']
df_inherited = df_inherited.drop(labels=cols_to_drop, axis=1, errors='ignore')

# 2. Impute missing values (using the same strategy as Notebook 03)
num_cols = ['LotFrontage', 'BedroomAbvGr', '2ndFlrSF', 'GarageYrBlt', 'MasVnrArea']
cat_cols = ['GarageFinish', 'BsmtFinType1', 'BsmtExposure']

for col in num_cols:
    if col in df_inherited.columns:
        df_inherited[col] = df_inherited[col].fillna(0) # Using 0 as a safe fallback for missing numericals in this subset

for col in cat_cols:
    if col in df_inherited.columns:
        df_inherited[col] = df_inherited[col].fillna('None')

print(f"Missing values remaining: {df_inherited.isnull().sum().sum()}")

Missing values remaining: 0


---
## 3. Feature Engineering
Here, we apply the transformations from Notebook 04. This includes Ordinal Encoding for categorical quality ratings and creating our time-based features (`HouseAge`, `RemodelAge`, and `GarageAge`) using 2010 as the baseline year, followed by dropping the original absolute year columns.

In [3]:
# 1. Ordinal Encoding
mapping_quality = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
mapping_exposure = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
mapping_fin_type = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
mapping_garage = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}

df_inherited['KitchenQual'] = df_inherited['KitchenQual'].map(mapping_quality)
df_inherited['BsmtExposure'] = df_inherited['BsmtExposure'].map(mapping_exposure)
df_inherited['BsmtFinType1'] = df_inherited['BsmtFinType1'].map(mapping_fin_type)
df_inherited['GarageFinish'] = df_inherited['GarageFinish'].map(mapping_garage)

# 2. Age Features (Reference year 2010 to match training data)
reference_year = 2010
df_inherited['HouseAge'] = reference_year - df_inherited['YearBuilt']
df_inherited['RemodelAge'] = reference_year - df_inherited['YearRemodAdd']
df_inherited['GarageAge'] = reference_year - df_inherited['GarageYrBlt']

columns_to_drop_years = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
df_inherited = df_inherited.drop(columns=columns_to_drop_years, axis=1)

# 3. Log Transformations (Matching the highly skewed features from Notebook 04)
skewed_features = [
    'LotArea', 'MasVnrArea', 'LotFrontage', 'OpenPorchSF',
    'BsmtFinSF1', 'TotalBsmtSF', '1stFlrSF', 'GrLivArea',
    'BsmtExposure', 'BsmtUnfSF', '2ndFlrSF'
]

for feature in skewed_features:
    if feature in df_inherited.columns:
        df_inherited[feature] = np.log1p(df_inherited[feature])

print("Feature engineering and log transformations completed!")
df_inherited.head()

Feature engineering and log transformations completed!


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,GarageArea,GarageFinish,GrLivArea,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,HouseAge,RemodelAge,GarageAge
0,6.799056,0.000000,2,0.693147,6.150603,3,5.602119,730.0,1,6.799056,...,9.360741,4.394449,0.000000,0.000000,6,5,6.783325,49,49,49.0
1,7.192934,0.000000,3,0.693147,6.828712,5,6.008813,312.0,1,7.192934,...,9.565775,4.406719,4.691348,3.610918,6,6,7.192934,52,52,52.0
2,6.834109,6.553933,3,0.693147,6.674561,6,4.927254,482.0,3,7.396335,...,9.534668,4.317488,0.000000,3.555348,5,5,6.834109,13,12,13.0
3,6.831954,6.520621,3,0.693147,6.401917,6,5.783825,470.0,3,7.380879,...,9.208238,4.369448,3.044522,3.610918,6,6,6.831954,12,12,12.0


In [4]:
import joblib

# Load the fitted scaler from the training phase
scaler_path = "outputs/ml_pipeline/predict_housing/v1/scaler.pkl"
scaler = joblib.load(scaler_path)

# Identify numerical columns to scale (matching the Notebook 04 logic)
features_to_scale = df_inherited.select_dtypes(include=['int64', 'float64']).columns
if 'SalePrice' in features_to_scale:
    features_to_scale = features_to_scale.drop('SalePrice')

# Apply the scaler using .transform() ONLY, to maintain the training distribution
df_inherited[features_to_scale] = scaler.transform(df_inherited[features_to_scale])

print("Feature scaling applied successfully to the inherited houses!")

Feature scaling applied successfully to the inherited houses!


---
## 4. Feature Filtering
Machine learning pipelines require the input features to be in the exact same order as they were during training. We will use the trained columns list to filter our engineered dataframe, preventing 'Silent Fails' during prediction.

In [5]:
# Load the train columns used by the model
train_columns_path = "outputs/ml_pipeline/predict_housing/v1/train_columns.pkl"
train_columns = joblib.load(train_columns_path)

# Filter the dataframe to ensure identical column order and prevent 'Silent Fails'
df_inherited_filtered = df_inherited.filter(train_columns)

print(f"Columns after filtering match training data exactly: {list(df_inherited_filtered.columns) == train_columns}")

Columns after filtering match training data exactly: True


---
## 5. Save the Engineered Dataset
With the data perfectly cleaned, engineered, and filtered, we save it to the `outputs` directory. The Streamlit app will load this processed file to instantly predict the portfolio's total value.

In [6]:
# Save to outputs so the Streamlit Dashboard can load it directly
output_dir = "outputs/datasets/collection"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "inherited_houses_engineered.csv")
df_inherited_filtered.to_csv(output_path, index=False)

print(f"Success! Engineered inherited houses saved to: {output_path}")

Success! Engineered inherited houses saved to: outputs/datasets/collection/inherited_houses_engineered.csv
